# Notebook 39 — Threat model: compression-aware attack-type selection against default-pruned detectors

**Attacker model.** The attacker knows only that the deployed detector is a CNN1D pruned with the default
per-layer 80% recipe. They do not know the specific trained model. From public knowledge (here: prune80
models from an attacker-side set of seeds), they estimate the *exposed set*: attack classes whose recall is
predictably destroyed. They then choose attack types from that set and send ordinary traffic of those
types, with no feature perturbation. The defender's cost is attribution failure: the attack is flagged, but
as the wrong class or the wrong family, so triage and response target the wrong thing.

**Evaluation.** Leave-two-seeds-out folds over the five independently trained baseline-mask pairs: the
exposed set is estimated on three attacker-side seeds and evaluated on two held-out deployed models the
attacker never saw. Strategies: random attack type (uniform over attack classes), exposed-set selection,
and an oracle that knows each deployed model's worst classes (upper bound, not available to an attacker).
Metrics on the selected traffic: exact-type misattribution, cross-family substitution, attack-to-benign miss
(silent evasion), and the benign false-alert rate the deployment generates on its own (alert-fatigue
pressure). The same evaluation is repeated against the two fixed recipes from Notebook 34 (first-layer
protected; global magnitude) and the dense baseline, to measure what the one-line defence removes.

**Gate (stated before running).** (i) On held-out default-pruned detectors, exposed-set selection yields
at least twice the misattribution rate of random selection; (ii) under either fixed recipe the same lift
falls below 1.2x. Inference only on saved checkpoints; GPU runtime for speed.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
from itertools import combinations
RECIPES = {'default_layerwise80': 'prune80_paired', 'protect_conv0': 'layerwise80_protect_conv0_paired', 'global80': 'global80_paired'}
EXPOSED_THRESHOLD = 0.20   # attack class is 'exposed' if attacker-side mean prune80 recall is below this

ARCH = 'cnn1d'; ARCH_KW = {'channels': (64, 128)}
print('recipes:', list(RECIPES), '| exposed threshold:', EXPOSED_THRESHOLD)

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Per-flow test predictions for every (recipe, seed), plus the dense baselines
fam_map = pd.read_csv(OUT / 'ciciot2023_alert_family_mapping.csv').set_index('fine_label')['alert_family'].to_dict()
def load_pruned(cell, seed, le):
    m = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
    ck = torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location=DEVICE, weights_only=False)
    m.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); return m.eval()

preds = {}   # (recipe, seed) -> (y_true, y_pred) as class-index arrays
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=ARCH_KW)
    yt, yp, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test'); preds[('dense', seed)] = (np.asarray(yt), np.asarray(yp))
    for recipe, cell in RECIPES.items():
        mp = load_pruned(cell, seed, le)
        yt, yp, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); preds[(recipe, seed)] = (np.asarray(yt), np.asarray(yp))
    print(f'seed {seed}: predictions collected')
classes = list(le.classes_); C = len(classes)
benign_idx = int(np.where(np.array(classes) == 'BenignTraffic')[0][0])
family_of = np.array([fam_map[c] for c in classes])
attack_idx = [i for i in range(C) if i != benign_idx]
print(f'{C} classes, benign index {benign_idx}, {len(attack_idx)} attack classes')

In [ ]:
# Per-class recall per (recipe, seed), and the attacker's exposed set from attacker-side seeds
def per_class_recall(yt, yp):
    return np.array([(yp[yt == c] == c).mean() if (yt == c).sum() else np.nan for c in range(C)])
recall = {k: per_class_recall(*v) for k, v in preds.items()}

def exposed_set(recipe, attacker_seeds):
    r = np.nanmean([recall[(recipe, s)] for s in attacker_seeds], axis=0)
    return [c for c in attack_idx if r[c] < EXPOSED_THRESHOLD]

def selection_metrics(yt, yp, chosen):
    # traffic of the chosen attack types, as the attacker would send it; metrics over those flows
    m = np.isin(yt, chosen); t, p = yt[m], yp[m]
    if len(t) == 0: return {'n_flows': 0}
    exact = (p == t); to_benign = (p == benign_idx); cross_family = (family_of[p] != family_of[t]) & ~to_benign
    return {'n_flows': int(len(t)), 'misattribution_rate': float(1 - exact.mean()), 'attack_to_benign_rate': float(to_benign.mean()),
            'cross_family_rate': float(cross_family.mean()), 'wrong_family_or_benign_rate': float((cross_family | to_benign).mean())}

rows, exposed_rows = [], []
for attacker_seeds in combinations(SEEDS, 3):
    held_out = [s for s in SEEDS if s not in attacker_seeds]
    for recipe in list(RECIPES) + ['dense']:
        exp_set = exposed_set('default_layerwise80', attacker_seeds)     # attacker assumes the DEFAULT recipe
        exposed_rows.append({'attacker_seeds': str(attacker_seeds), 'exposed_set': ';'.join(classes[c] for c in exp_set), 'n_exposed': len(exp_set)})
        for seed in held_out:
            yt, yp = preds[(recipe, seed)]
            oracle = [c for c in attack_idx if recall[(recipe, seed)][c] < EXPOSED_THRESHOLD]
            for strategy, chosen in (('random_attack_type', attack_idx), ('exposed_set', exp_set), ('oracle_per_model', oracle)):
                # random strategy = uniform over attack TYPES: weight each type equally rather than by prevalence
                if strategy == 'random_attack_type':
                    per_type = [selection_metrics(yt, yp, [c]) for c in attack_idx]; per_type = [d for d in per_type if d['n_flows'] > 0]
                    met = {k: float(np.mean([d[k] for d in per_type])) for k in per_type[0] if k != 'n_flows'}; met['n_flows'] = int(sum(d['n_flows'] for d in per_type))
                else:
                    met = selection_metrics(yt, yp, chosen)
                rows.append({'attacker_seeds': str(attacker_seeds), 'held_out_seed': seed, 'recipe': recipe, 'strategy': strategy, 'n_chosen_types': len(chosen), **met})
fold = pd.DataFrame(rows); fold.to_csv(OUT / 'threat_model_fold_results.csv', index=False)
pd.DataFrame(exposed_rows).drop_duplicates().to_csv(OUT / 'threat_model_exposed_sets.csv', index=False)
print('exposed sets by attacker-side seeds:'); print(pd.DataFrame(exposed_rows).drop_duplicates().to_string(index=False))

In [ ]:
# Summary: mean over folds and held-out models; lift = exposed-set misattribution / random misattribution
summ = fold.groupby(['recipe', 'strategy']).agg(misattribution=('misattribution_rate', 'mean'), attack_to_benign=('attack_to_benign_rate', 'mean'),
        cross_family=('cross_family_rate', 'mean'), wrong_family_or_benign=('wrong_family_or_benign_rate', 'mean'), n_types=('n_chosen_types', 'mean')).reset_index()
lift = {}
for recipe in list(RECIPES) + ['dense']:
    r = summ[summ.recipe == recipe].set_index('strategy')
    lift[recipe] = float(r.loc['exposed_set', 'misattribution'] / r.loc['random_attack_type', 'misattribution']) if 'exposed_set' in r.index and r.loc['random_attack_type', 'misattribution'] > 0 else float('nan')
summ['exposed_lift_vs_random'] = summ.recipe.map(lift)
summ.to_csv(OUT / 'threat_model_summary.csv', index=False)
print(summ.round(4).to_string(index=False))

# alert-fatigue pressure the deployment creates on its own: benign flows flagged as attacks, per recipe
fatigue = []
for recipe in list(RECIPES) + ['dense']:
    rates = [float((preds[(recipe, s)][1][preds[(recipe, s)][0] == benign_idx] != benign_idx).mean()) for s in SEEDS]
    fatigue.append({'recipe': recipe, 'benign_to_attack_rate_mean': np.mean(rates), 'sd': np.std(rates, ddof=1)})
fatigue = pd.DataFrame(fatigue); fatigue.to_csv(OUT / 'threat_model_alert_fatigue.csv', index=False)
print(); print(fatigue.round(4).to_string(index=False))

verdict = pd.DataFrame([
 {'criterion': 'i_default_exposed_lift_ge_2x', 'value': round(lift['default_layerwise80'], 3), 'pass': bool(lift['default_layerwise80'] >= 2.0)},
 {'criterion': 'ii_protect_conv0_lift_lt_1.2x', 'value': round(lift['protect_conv0'], 3), 'pass': bool(lift['protect_conv0'] < 1.2)},
 {'criterion': 'ii_global80_lift_lt_1.2x', 'value': round(lift['global80'], 3), 'pass': bool(lift['global80'] < 1.2)},
 {'criterion': 'ref_dense_lift', 'value': round(lift['dense'], 3), 'pass': ''},
])
print(); print(verdict.to_string(index=False))
print('\nCompression-aware selection is a predictable attribution-evasion surface removed by the fixed recipes:', bool(verdict[verdict['pass'] != '']['pass'].astype(bool).all()))
verdict.to_csv(OUT / 'threat_model_gate_verdict.csv', index=False)
write_json(OUT / 'threat_model_environment.json', {'recipes': RECIPES, 'exposed_threshold': EXPOSED_THRESHOLD, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/39_threat_model_attack_type_selection.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/threat_model_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 39: threat model - compression-aware attack-type selection with leave-two-seeds-out exposed sets; lift under default vs fixed recipes; alert-fatigue pressure'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)